# CLOY Processing

This notebook is for you to work with the CLOY data folder.

In [2]:
%pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 29.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 38.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install textblob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [textblob]7/8 [textblob]le]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import math
import pandas as pd
from textblob import TextBlob
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [63]:
threads=[]
with open("data/RedditExplorer/cloy_chronological.jsonl") as f:
    for line in f:
        threads.append(json.loads(line))
print(len(threads))

3075


In [64]:
for t in threads:
    if t["post_id"]=="1d6vjw4":
        thread=t

In [65]:
def get_comment_text(comments):
    text=""
    for comment in comments:
        text=text+" "+comment["comment_body"]
        if "replies" in comment:
            text=text+get_comment_text(comment["replies"])
    return text

def get_thread_text(thread):
    return thread["title"]+" "+thread["selftext"]+get_comment_text(thread["comments"])

print(get_thread_text(thread)[:500])

If CLOY is #1, what's your #2? Clearly we all have excellent taste, so I'm wondering how that extends past CLOY?

I've seen some people list a bunch of different shows they like as well, and I've taken note of a few, but I'm really interested in seeing if people can just name one other show that really stands out for them like CLOY does.

For me the only thing that's come close so far is **Extraordinary Attorney Woo** - very different of course, but just struck the right notes with me.

So! What


In [66]:
vectors={}
titles={}
for thread in threads:
    vectors[thread["post_id"]]=TextBlob(get_thread_text(thread)).word_counts
    titles[thread["post_id"]]=thread["title"]

print(vectors["1d6vjw4"])

defaultdict(<class 'int'>, {'if': 6, 'cloy': 29, 'is': 49, '1': 6, 'what': 6, 's': 29, 'your': 2, '2': 11, 'clearly': 1, 'we': 2, 'all': 10, 'have': 13, 'excellent': 3, 'taste': 2, 'so': 24, 'i': 70, 'm': 7, 'wondering': 1, 'how': 4, 'that': 22, 'extends': 1, 'past': 1, 've': 11, 'seen': 5, 'some': 2, 'people': 4, 'list': 4, 'a': 40, 'bunch': 1, 'of': 47, 'different': 5, 'shows': 4, 'they': 4, 'like': 6, 'as': 9, 'well': 5, 'and': 59, 'taken': 1, 'note': 2, 'few': 3, 'but': 23, 'really': 8, 'interested': 1, 'in': 23, 'seeing': 2, 'can': 9, 'just': 7, 'name': 4, 'one': 17, 'other': 4, 'show': 7, 'stands': 1, 'out': 6, 'for': 39, 'them': 3, 'does': 4, 'me': 29, 'the': 83, 'only': 5, 'thing': 1, 'come': 3, 'close': 3, 'far': 2, 'extraordinary': 4, 'attorney': 5, 'woo': 5, 'very': 8, 'course': 1, 'struck': 1, 'right': 3, 'notes': 1, 'with': 21, 'yours': 1, 'king': 4, 'land': 1, 'about': 3, 'to': 34, 'finish': 1, 'it': 55, 'my': 41, 'wife': 2, 'says': 1, '’': 18, 'her': 2, 'best': 4, 'glad'

In [67]:
def cosim(v1, v2):
    dot=0
    for word in v1:
        if word in v2:
            dot=dot+v1[word]*v2[word]

    length1=0
    for word in v1:
        length1=length1+v1[word]**2

    length2=0
    for word in v2:
        length2=length2+v2[word]**2

    if length1==0 or length2==0:
        return 0
    return dot/(math.sqrt(length1)*math.sqrt(length2))

In [69]:
N=1000

ids=list(vectors.keys())[:N]
if "1d6vjw4" not in ids:
    ids.append("1d6vjw4")

similarity={}
for id1 in ids:
    similarity[id1]={}

for i in range(len(ids)):
    for j in range(i, len(ids)):
        score=cosim(vectors[ids[i]], vectors[ids[j]])
        similarity[ids[i]][ids[j]]=score
        similarity[ids[j]][ids[i]]=score
sim_df=pd.DataFrame(similarity)

In [70]:
def top_5_similar(thread_id):
    print(thread_id, titles[thread_id])
    print()
    scores=sim_df[thread_id].drop(thread_id)
    top5=scores.sort_values(ascending=False).head(5)
    for other_id in top5.index:
        print(other_id, titles[other_id])

top_5_similar("1d6vjw4")

1d6vjw4 If CLOY is #1, what's your #2?

iufpmn CLOY set my expectations to high as my first kdrama
k8wcoc so....what to watch next? Tried "something in the rain"...too sappy for me; need some action, some comedy, some chemistry and some suspense!
kpu73v My thoughts as this was my first kdrama
iy5qeu Withdrawal, anyone?
hwchxk Moving on from CLOY?


In [71]:
from nltk.corpus import stopwords
stop_words=stopwords.words("english")

vectors_no_stop={}
for thread_id in vectors:
    new_vector={}
    for word in vectors[thread_id]:
        if word not in stop_words:
            new_vector[word]=vectors[thread_id][word]
    vectors_no_stop[thread_id]=new_vector

In [73]:
def top_5_similar_no_stop(thread_id):
    print(thread_id, titles[thread_id])
    print()
    scores={}
    for other_id in ids:
        if other_id!=thread_id:
            scores[other_id]=cosim(vectors_no_stop[thread_id], vectors_no_stop[other_id])
    top5=sorted(scores, key=scores.get, reverse=True)[:5]
    for other_id in top5:
        print(other_id, titles[other_id])

top_5_similar_no_stop("1d6vjw4")

1d6vjw4 If CLOY is #1, what's your #2?

iufpmn CLOY set my expectations to high as my first kdrama
k8wcoc so....what to watch next? Tried "something in the rain"...too sappy for me; need some action, some comedy, some chemistry and some suspense!
hwchxk Moving on from CLOY?
kpu73v My thoughts as this was my first kdrama
i8482w Just finished watching Cloy again!


#####################

In [ ]:
with open("data/RedditExplorer/cloy_chronological.jsonl", encoding="utf-8") as f:
    for line in f:
        threads.append(json.loads(line))
print("Number of threads:", len(threads))

Number of threads: 3075


In [38]:
for t in threads:
    if t["post_id"] == "1d6vjw4":
        thread = t

print(thread.keys())
print(thread["title"])
print(thread["comments"][0].keys())
print(thread["comments"][0]["comment_body"])
print(thread["comments"][0]["replies"][0]["comment_body"])

dict_keys(['post_id', 'timestamp', 'author', 'subreddit', 'title', 'selftext', 'url', 'score', 'permalink', 'comments'])
If CLOY is #1, what's your #2?
dict_keys(['comment_id', 'parent_id', 'author', 'comment_body', 'score', 'created_utc', 'replies'])
King the land
About to finish it, my wife says that’s her best


In [9]:
import os

for folder, subfolders, files in os.walk("."):
    if "cloy_chronological.jsonl" in files:
        print(os.path.join(folder, "cloy_chronological.jsonl"))

./data/RedditExplorer/cloy_chronological.jsonl


In [39]:
def get_comment_text(comments):
    text = ""
    for comment in comments:
        text = text + " " + comment["comment_body"]
        if "replies" in comment:
            text = text + get_comment_text(comment["replies"])
    return text

def get_thread_text(thread):
    return thread["title"] + " " + thread["selftext"] + get_comment_text(thread["comments"])

print(get_thread_text(thread)[:500])

If CLOY is #1, what's your #2? Clearly we all have excellent taste, so I'm wondering how that extends past CLOY?

I've seen some people list a bunch of different shows they like as well, and I've taken note of a few, but I'm really interested in seeing if people can just name one other show that really stands out for them like CLOY does.

For me the only thing that's come close so far is **Extraordinary Attorney Woo** - very different of course, but just struck the right notes with me.

So! What


In [40]:
vectors= {}
titles= {}
for thread in threads:
    vectors[thread["post_id"]] = TextBlob(get_thread_text(thread)).word_counts
    titles[thread["post_id"]] = thread["title"]

print(vectors["1d6vjw4"])

defaultdict(<class 'int'>, {'if': 6, 'cloy': 29, 'is': 49, '1': 6, 'what': 6, 's': 29, 'your': 2, '2': 11, 'clearly': 1, 'we': 2, 'all': 10, 'have': 13, 'excellent': 3, 'taste': 2, 'so': 24, 'i': 70, 'm': 7, 'wondering': 1, 'how': 4, 'that': 22, 'extends': 1, 'past': 1, 've': 11, 'seen': 5, 'some': 2, 'people': 4, 'list': 4, 'a': 40, 'bunch': 1, 'of': 47, 'different': 5, 'shows': 4, 'they': 4, 'like': 6, 'as': 9, 'well': 5, 'and': 59, 'taken': 1, 'note': 2, 'few': 3, 'but': 23, 'really': 8, 'interested': 1, 'in': 23, 'seeing': 2, 'can': 9, 'just': 7, 'name': 4, 'one': 17, 'other': 4, 'show': 7, 'stands': 1, 'out': 6, 'for': 39, 'them': 3, 'does': 4, 'me': 29, 'the': 83, 'only': 5, 'thing': 1, 'come': 3, 'close': 3, 'far': 2, 'extraordinary': 4, 'attorney': 5, 'woo': 5, 'very': 8, 'course': 1, 'struck': 1, 'right': 3, 'notes': 1, 'with': 21, 'yours': 1, 'king': 4, 'land': 1, 'about': 3, 'to': 34, 'finish': 1, 'it': 55, 'my': 41, 'wife': 2, 'says': 1, '’': 18, 'her': 2, 'best': 4, 'glad'

In [50]:
lengths = {}
for thread_id in vectors:
    total = 0
    for word in vectors[thread_id]:
        total = total + vectors[thread_id][word] ** 2
    lengths[thread_id] = math.sqrt(total)

In [51]:
def cosim(id1, id2):
    v1 = vectors[id1]
    v2 = vectors[id2]
    dot = 0
    for word in v1:
        if word in v2:
            dot = dot + v1[word] * v2[word]
    if lengths[id1] == 0 or lengths[id2] == 0:
        return 0
    return dot / (lengths[id1] * lengths[id2])

In [ ]:
N=1000
ids = list(vectors.keys())[:N]
if "1d6vjw4" not in ids:
    ids.append("1d6vjw4")
similarity = {}
for id1 in ids:
    similarity[id1] = {}

for i in range(len(ids)):
    for j in range(i, len(ids)):
        score = cosim(ids[i], ids[j])
        similarity[ids[i]][ids[j]] = score
        similarity[ids[j]][ids[i]] = score
sim_df = pd.DataFrame(similarity)

In [55]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
stop_words = stopwords.words("english")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [31]:
for t in threads:
    if t["post_id"]=="1d6vjw4":
        thread=t
comment = thread["comments"][0]
reply = comment["replies"][0]
reply_to_reply = reply["replies"][0]

In [13]:
def get_comment_text(comments):
    text = ""
    for comment in comments:
        if comment["comment_body"] not in SKIP:
            text = text + " " + comment["comment_body"]
        replies = comment.get("replies", [])
        text = text + " " + get_comment_text(replies)
    return text
def get_thread_text(thread):
    text = ""
    if not thread["title"].startswith("DUMMY Thread"):
        text = thread["title"]
    if thread["selftext"] not in SKIP:
        text = text + " " + thread["selftext"]
    text = text + " " + get_comment_text(thread["comments"])
    return text
print(get_thread_text(thread)[:500])

If CLOY is #1, what's your #2? Clearly we all have excellent taste, so I'm wondering how that extends past CLOY?

I've seen some people list a bunch of different shows they like as well, and I've taken note of a few, but I'm really interested in seeing if people can just name one other show that really stands out for them like CLOY does.

For me the only thing that's come close so far is **Extraordinary Attorney Woo** - very different of course, but just struck the right notes with me.

So! What


In [36]:
SKIP = ["",
    "[deleted]",
    "[removed]",
    "This placeholder thread was generated to preserve historical comments whose parent post data is missing.",]

In [25]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [26]:
def text_to_vector(text, stop_words=[]):
    words= TextBlob(text.lower()).words
    vector={}
    for word in words:
        if word.isalpha() and word not in stop_words:
            vector[word]=vector.get(word, 0) + 1
    return vector
vectors={}
titles={}
for thread in threads:
    thread_id = thread["post_id"]
    vectors[thread_id] = text_to_vector(get_thread_text(thread))
    titles[thread_id] = thread["title"]
print(len(vectors), "vectors")
print(vectors["1d6vjw4"])

3075 vectors
{'if': 6, 'cloy': 29, 'is': 49, 'what': 6, 'your': 2, 'clearly': 1, 'we': 2, 'all': 10, 'have': 13, 'excellent': 3, 'taste': 2, 'so': 24, 'i': 70, 'wondering': 1, 'how': 4, 'that': 22, 'extends': 1, 'past': 1, 'seen': 5, 'some': 2, 'people': 4, 'list': 4, 'a': 40, 'bunch': 1, 'of': 47, 'different': 5, 'shows': 4, 'they': 4, 'like': 6, 'as': 9, 'well': 5, 'and': 59, 'taken': 1, 'note': 2, 'few': 3, 'but': 23, 'really': 8, 'interested': 1, 'in': 23, 'seeing': 2, 'can': 9, 'just': 7, 'name': 4, 'one': 17, 'other': 4, 'show': 7, 'stands': 1, 'out': 6, 'for': 39, 'them': 3, 'does': 4, 'me': 29, 'the': 83, 'only': 5, 'thing': 1, 'come': 3, 'close': 3, 'far': 2, 'extraordinary': 4, 'attorney': 5, 'woo': 5, 'very': 8, 'course': 1, 'struck': 1, 'right': 3, 'notes': 1, 'with': 21, 'yours': 1, 'king': 4, 'land': 1, 'about': 3, 'to': 34, 'finish': 1, 'it': 55, 'my': 41, 'wife': 2, 'says': 1, 's': 9, 'her': 2, 'best': 4, 'glad': 1, 'you': 13, 'guys': 1, 'liked': 2, 'ktl': 2, 'such': 2,

In [ ]:
def cosine_sim(v1, v2):
    dot_product = 0
    for word in v1:
        if word in v2:
            dot_product = dot_product + v1[word] * v2[word]

    length1 = math.sqrt(sum(count ** 2 for count in v1.values()))
    length2 = math.sqrt(sum(count ** 2 for count in v2.values()))

    if length1 == 0 or length2 == 0:
        return 0
    return dot_product/(length1 * length2)

In [30]:
def build_similarity_table(vectors):
    ids = list(vectors.keys())
    similarity = {}
    for thread_id in ids:
        similarity[thread_id] = {}
    for i in range(len(ids)):
        for j in range(i, len(ids)):
            score = cosine_sim(vectors[ids[i]], vectors[ids[j]])
            similarity[ids[i]][ids[j]] = score
            similarity[ids[j]][ids[i]] = score
        if i % 500 == 0:
            print(i, "threads done")

    return pd.DataFrame(similarity)

In [ ]:
def top_5_similar(thread_id, sim_df):
    print("Threads most similar to", thread_id, "-", titles[thread_id])
    scores = sim_df[thread_id].drop(thread_id)
    top5 = scores.sort_values(ascending=False).head(5)
    for other_id in top5.index:
        print(other_id, "|", titles[other_id], "|", round(top5[other_id], 3))

In [ ]:
top_5_similar("1d6vjw4", sim_df)

Threads most similar to 1d6vjw4 - If CLOY is #1, what's your #2?


KeyError: '1d6vjw4'

In [ ]:
def most_common_words(vectors, n=20):
    totals = {}
    for vector in vectors.values():
        for word in vector:
            totals[word] = totals.get(word, 0) + vector[word]
    return sorted(totals.items(), key=lambda pair: pair[1], reverse=True)[:n]

print(most_common_words(vectors))